# Preparing metadata from KCL

Metadata were extracted from DICOM files using existing code that uses pydicom to read selected fields into a dataframe with each field in a column. The fields were taken from the DicomStandardReference spreadsheet, creating one list for CT and one for MR. Any fields not already collected were indexed. They were then pivoted to a long form (key/value pairs) to match the evaluator requirements.


In [ ]:
import csv
from pathlib import Path
import pandas as pd

meta_dir_ct = Path('/nfs/project/WellcomeHDN/kch-ct/metadata/dicom_heterogeneity')
meta_dir_mr = Path('/nfs/project/WellcomeHDN/kch-mr/metadata/dicom_heterogeneity')
modality = 'MR'

# Load old dicom index
if modality == 'CT':
    meta_dir = meta_dir_ct
    di_old = pd.read_csv(meta_dir_ct / '..' / 'dicom_index_ct_picks.csv', dtype=str).rename(columns={'dcm_file': 'dicom_filepath'})
    di_old = di_old.query('valid == "True"')
elif modality == 'MR':
    meta_dir = meta_dir_mr
    di_old = pd.read_parquet(meta_dir_mr / '..' / 'merge_kch_midi' / 'dicom_index_mr_merged.parquet')
else:
    raise ValueError(f'Unknown modality: {modality}')

In [ ]:
# Load new dicom index
di_new = pd.read_csv(meta_dir / 'dicom_index.csv', dtype=str)
print(di_new.shape)

In [ ]:
# The two dicom indexes took from the same directories, but might have picked different files, so convert .dcm filepath to parent directory
di_old['dicom_path'] = di_old['dicom_filepath'].apply(lambda x: str(Path(x).parent))
di_new['dicom_path'] = di_new['dicom_filepath'].apply(lambda x: str(Path(x).parent))
# Check everything matches
print(di_old['dicom_path'].isin(di_new['dicom_path']).sum())
print(di_new['dicom_path'].isin(di_old['dicom_path']).sum())
print(di_old['dicom_path'].duplicated().sum())
# Each dataframe has the same set of unique identifiers

In [ ]:
# Show study and series counts for each year
study_counts = di_old.drop_duplicates(subset='StudyInstanceUID')['StudyDate'].str[:4].value_counts().sort_index().rename('studies')
series_counts = di_old.drop_duplicates(subset='SeriesInstanceUID')['StudyDate'].str[:4].value_counts().sort_index().rename('series')
counts = pd.concat([study_counts, series_counts], axis=1)
counts.to_csv(meta_dir / f'study_series_counts_by_year_{modality}.csv')
counts


In [ ]:
eval_fields = pd.read_csv(meta_dir / f'dicom_fields_dicom_heterogeneity_{modality}.txt', header=None)
print(f'Using {len(eval_fields)} fields for evaluation')

di_fields1 = di_old[['dicom_path'] + di_old.columns[di_old.columns.isin(eval_fields[0])].tolist()]
di_fields2 = di_new[['dicom_path'] + di_new.columns[di_new.columns.isin(eval_fields[0])].tolist()]
print(f'Using {di_fields1.shape[1]-1} fields from old index and {di_fields2.shape[1]-1} fields from new index')
di_fields = di_fields1.merge(di_fields2, on='dicom_path', how='outer')
print(f'Total fields after merging: {di_fields.shape[1]-1}')

In [ ]:
di_fields_long = di_fields.melt(id_vars='dicom_path', var_name='Keyword', value_name='Value')
di_fields_long = di_fields_long.loc[di_fields_long['Value'].notna(), :]
from pydicom.tag import Tag
di_fields_long['Tag'] = di_fields_long['Keyword'].apply(lambda x: str(Tag(x)))
di_fields_long[['Tag', 'Keyword']].value_counts(dropna=False)

In [ ]:
# Merge back other used fields
df_metadata = di_fields_long.merge(di_old[['dicom_path', 'StudyInstanceUID', 'SeriesInstanceUID', 'Manufacturer', 'ManufacturerModelName']], on='dicom_path', how='left')
df_metadata['IOD'] = f'{modality} Image IOD'
df_metadata.rename(columns={
    'StudyInstanceUID': 'study_id',
    'SeriesInstanceUID': 'series_id',
    'ManufacturerModelName': 'ScannerModel',
    'dicom_path': 'file_id',
    'Keyword': 'AttributeName',
}, inplace=True)
df_metadata = df_metadata[['IOD', 'study_id', 'series_id', 'file_id', 'Manufacturer', 'ScannerModel', 'Tag', 'AttributeName', 'Value']]
df_metadata.head()

In [ ]:
df_metadata.to_csv(meta_dir / f'df_metadata_kch-{modality.lower()}.csv', index=False, quoting=csv.QUOTE_NONNUMERIC)